# Day 15 · Inside out · Study D: Where AI Is at Risk

Counties shaded by **FEMA's National Risk Index**, with the data centers on top. Something here is upside down:
billion-dollar campuses going up in the counties most exposed to natural hazards.

Pick the hazard in step 1: overall risk, wildfire, heat wave, drought, riverine or coastal flooding, hurricane, tornado.
**Needs:** the NRI county CSV in `data/raw/` (see README). Census county shapes download automatically.

## 1 · Settings

In [ ]:
DAY      = 15
PROMPT   = "Inside out"
STUDY    = "D"
SLUG     = "insideout-risk-ai"
NOTEBOOK = "day15_insideout_risk_ai.ipynb"

HAZARD    = "RISK"      # "RISK" overall | "WFIR" wildfire | "HWAV" heat wave | "DRGT" drought | "RFLD" river flood | "CFLD" coastal flood | "HRCN" hurricane | "TRND" tornado
TITLE     = "Where AI Is at Risk"
HIGHLIGHT = "AI"
SUBTITLE  = None        # None = filled in from the numbers in step 3
CREDITS   = ["im3", "epoch", "nri", "census"]

REGION   = "us48"
TOOL     = "mapbox"
MAPBOX_STYLE = "dark"
PROJECTION   = "globe"

## 2 · Load

In [ ]:
import pandas as pd
import mapkit as mk
from mapkit import ai, publish

dcs = ai.clip(ai.master(), REGION)
ex, big = dcs[dcs.kind == "existing"], dcs[dcs.kind == "ai_campus"]
print(f"{len(ex):,} existing data centers · {len(big)} AI campuses")
counties = ai.load_risk_index()
hz = ai.NRI_HAZARDS[HAZARD]
score, rating = f"{HAZARD}_score", f"{HAZARD}_rating"
print(f"{len(counties):,} counties · hazard: {hz}")
counties[rating].value_counts()

## 3 · Analysis: how many AI campuses sit in high-risk counties?

In [ ]:
dcs = ai.attach(dcs, counties[[score, rating, "geometry"]], [score, rating])
ai.update_master(dcs, [score, rating], about=f"FEMA National Risk Index for the county: {hz}",
                 source="FEMA National Risk Index", day=DAY)

ex, big = dcs[dcs.kind == "existing"], dcs[dcs.kind == "ai_campus"]
coming = big[big.status != "Operating"]
is_high = lambda d: d[rating].isin(["Very High", "Relatively High"])
pct = lambda d: round(100 * is_high(d).sum() / max(len(d), 1))

print(f"AI campuses in counties rated high for {hz.lower()}: {is_high(big).sum()} of {len(big)} ({pct(big)}%)")
print(f"  ...of those still being built or expanding: {is_high(coming).sum()} of {len(coming)}")
print(f"Existing data centers in high-rated counties: {pct(ex)}%")
print("\nAI campuses by county rating:")
print(big[rating].fillna("No rating").value_counts().to_string())

if SUBTITLE is None:
    SUBTITLE = f"{is_high(big).sum()} of {len(big)} new AI campuses sit in counties rated high for {hz.lower()}"
ABOUT = (f"Counties are shaded by FEMA's National Risk Index for {hz.lower()}, which combines how often hazards hit, "
         "what is exposed, and how well communities can recover. Brighter means higher risk. Dots and circles are the "
         "same data centers and AI campuses as in Where AI Gets Built.")
print("\nSubtitle:", SUBTITLE)

ramp = mk.style.LOOK["risk_ramp"]                   # dark -> bright purple, low -> high
risk_colors = dict(zip(reversed(ai.NRI_RATINGS), ramp))
overlays = [mk.Layer(counties[counties[rating].isin(ai.NRI_RATINGS)], (hz if HAZARD == "RISK" else f"{hz} risk") + " by county", column=rating,
                     order=ai.NRI_RATINGS, colors=risk_colors, opacity=0.6, tooltip="county_name",
                     popup={rating: "Risk rating", "state_abb": "State"})]
extra_popup = {rating: "County risk rating"}

## 4 · Map

In [ ]:
layers = overlays + ai.base_layers(dcs, extra_popup=extra_popup)    # overlay underneath, the cast on top
bounds = ai.REGIONS[REGION] if isinstance(REGION, str) else None
credits = mk.style.credits(*CREDITS)

if TOOL == "mapbox":
    m = mk.mapbox(layers, title=TITLE, subtitle=SUBTITLE, about=ABOUT, sources=credits, series="ai",
                  bounds=bounds, style=MAPBOX_STYLE, projection=PROJECTION, highlight=HIGHLIGHT)
elif TOOL == "folium":
    m = mk.interactive(layers, day=DAY, prompt=PROMPT, title=TITLE, subtitle=SUBTITLE, sources=credits,
                       series="ai", bounds=bounds)
else:
    m = mk.static(layers, day=DAY, prompt=PROMPT, title=TITLE, subtitle=SUBTITLE, sources=credits,
                  series="ai", bounds=bounds, crs="EPSG:5070")
m if TOOL == "folium" else None

## 5 · Export + caption numbers

In [ ]:
publish.save(m, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE, subtitle=SUBTITLE, series="ai", study=STUDY,
             tool={"mapbox": "Python + Mapbox", "folium": "Python + Folium", "static": "Python"}[TOOL],
             notebook=NOTEBOOK, credits=mk.style.credits(*CREDITS, full=True))

print(f"""
CAPTION NUMBERS
- {is_high(big).sum()} of {len(big)} AI campuses ({pct(big)}%) sit in counties rated high for {hz.lower()}
- {is_high(coming).sum()} of the {len(coming)} still being built or expanding
""")